In [ ]:
using Combinatorics, ITensors, JLD2, Yao, ITensorMPS, Plots, NPZ
include("../../../src/apply_gate_as_mpo.jl");
include("../../../src/gates_utils.jl");
#Script with operations w/ majoranas
include("../../../src/majo_machinery.jl");
#Script with possible initial configurations
include("../../../src/initial_circuits.jl");
#Script with potential quantum gates
include("../../../src/gates_circuit.jl");

In [ ]:
@show Threads.nthreads()

In [ ]:
#number qubits 
nq = 30
#we want the products in module Bm
m = 1
#how many points obtained as different angles t
num_points_1 = 50
num_points_2 = 50
t_max = 3.0
#input state parameterization
n_params = 2
#obtain majoranas and coefficients
list_majoranas = majorana_products(nq, m)
len_maj = length(list_majoranas);
#Operators out of list_majoranas
list_maj_op = [ops for (_, ops) in list_majoranas]
#Coefficients out of list_majoranas
coeff_maj_op = [real(coef) for (coef,_) in list_majoranas]
# For the FLO we have random coefficients 
tc = randn(len_maj); random_coeffs = tc/norm(tc);

In [ ]:
#Function to perform multiplication coeff*op
function make_operator(sites::Vector{Index{Int64}}, op::Symbol, index::Int)
    ITensor(paulis[op], sites[index]', sites[index])
end

#Initialize sites for tensors
sites_maj = siteinds("Qubit", nq)

# Generate paulis out of majoranas
paulis_maj = [MPO([make_operator(sites_maj, list_maj_op[i][j], j)
               for j in 1:nq]) for i in 1:len_maj];

# Include coeffcients to be actual majoranas, not just Paulis
op_maj = paulis_maj .* coeff_maj_op;

# Generate FLO 
op_flo_ = paulis_maj .* coeff_maj_op .* random_coeffs;

# Select just a bunch of terms out of FLO 
num_terms = len_maj
indices = randperm(len_maj)[1:num_terms]
op_flo = [op_flo_[i] for i in indices];


In [ ]:
#Write indices and coeffs in case you want to reuse them in python

npzwrite("../data/indices_BO_$(nq)q.npy", indices)
npzwrite("../data/coeffs_BO_$(nq)q.npy", random_coeffs[indices])

In [ ]:
# Function to get contractions gate |> state

function get_state!(gates, state::MPS, sites::Vector{Index{Int}}, indices::Vector{Vector{Int}})
    final_state = copy(state)
    
    for (j, gate) in enumerate(gates)
        inds = indices[j]
        it = length(size(gate)) == 2 ?
             ITensor(gate, sites[inds[1]]', sites[inds[1]]) :
             ITensor(gate, sites[inds[1]]', sites[inds[2]]', sites[inds[1]], sites[inds[2]])

        final_state = noprime(apply_mpo_gate(final_state, gate_to_mpo(it), inds));
    end

    return final_state
end

In [ ]:
# Function to group observable in case there is a tradeoff in which increasing χ and reducing #operators is faster

function group_mpos(mpos::Vector{MPO}, size_g::Int; maxbond::Int)
    ng = ceil(Int, length(mpos) / size_g)
    grouped = Vector{MPO}(undef, ng)

    for i in 1:ng
        range = (i-1)*size_g + 1 : min(i*size_g, length(mpos))
        grouped[i] = truncate(sum(mpos[range]), maxdim = maxbond)
    end

    return grouped
end

In [ ]:
#Initialize zero state 
zero_state = fill("0",nq)
MPS_zero = productMPS(sites_maj, zero_state);

# Create 1D arrays for t1 and t2
t1_list = collect(range(0, t_max, num_points_1))
t2_list = collect(range(0, t_max, num_points_2))

# Create a grid of shape (num_points, num_points, 2)
t_list = Array{Float64}(undef, num_points_1, num_points_2, 2)
for i in 1:num_points_1
    for j in 1:num_points_2
        t_list[i, j, 1] = t1_list[i]
        t_list[i, j, 2] = t2_list[j]
    end
end
npzwrite("../data/t_list_BO_$(nq)q.npy", t_list)

#Layer in quantum circuit
nlayers = 1

#Maximum bond dimension initial state
#If after given layer χ is greater than maxbond then truncate to χ and get state out of circuit
maxbond = 64

#Types of circuits for initialization
state_generators = [
    random_rotations_with_entanglement_rotations_state_circuit,
    random_rotations_with_entanglement_state_circuit,
    simple_extent_state_circuit,
    first_qubit_rotation_state_circuit,
    random_rotations_state_circuit,
    random_fermionic_gaussian_circuit
];

#Initial state after applying geenrator circuit 
initial_state = Matrix{Any}(undef, num_points_1, num_points_2)
for i in 1:num_points_1
    for j in 1:num_points_2
        t1 = t_list[i, j, 1]
        t2 = t_list[i, j, 2]
        initial_state[i, j] = state_generators[1](MPS_zero, sites_maj, t1, t2, nlayers, maxbond; seed = 42)
    end
end



In [ ]:
# A fuction to compute individual overlaos: easier to track down

function calculate_contributions(op_list::Vector{MPO}, st::MPS)

    contributions = [noprime(op * st) for op in op_list];

    return contributions

end

In [ ]:
# Select observable that is full observable (up to chosen number of terms)

observable_flo = op_flo;

# Select observable that is the baiss elements in the module

observable_maj = op_maj;

In [ ]:
# Compute outputs
contributions_mps_flo = Matrix{Any}(undef, num_points_1, num_points_2)
final_state_flo = Matrix{Any}(undef, num_points_1, num_points_2)
outputs = zeros(Float64, num_points_1, num_points_2)

Threads.@threads for i in 1:num_points_1
    for j in 1:num_points_2
        #println("Processing sample $i on thread $(Threads.threadid())")
        contributions_mps_flo[i, j] = calculate_contributions(observable_flo, initial_state[i, j])
        final_state_flo[i, j] = sum(contributions_mps_flo[i, j])
        outputs[i, j] = real(inner(initial_state[i, j], final_state_flo[i, j]))
    end
end

In [ ]:
outputs 

In [ ]:
# Compute feature vectors

feat_vectors = Matrix{Vector{Float64}}(undef, num_points_1, num_points_2)

Threads.@threads for i in 1:num_points_1
    #println("Processing sample $i on thread $(Threads.threadid())")
    for j in 1:num_points_2
        init_s = initial_state[i, j]
        #println("Processing sample $j on thread $(Threads.threadid())")
        vec = Vector{Float64}(undef, len_maj)
        for k in 1:len_maj
            vec[k] = real(inner(init_s, observable_maj[k], init_s))
        end
        feat_vectors[i, j] = vec
    end
end

In [ ]:
npzwrite("../data/outputs_BO_$(nq)q.npy", outputs)
array_data = permutedims(reshape(hcat(reshape(feat_vectors, :)...), len_maj, num_points_1, num_points_2), (2, 3, 1))  # convert the matrix of vectors to a 3D array
npzwrite("../data/feat_vec_BO_$(nq)q.npy", array_data)